In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np

In [16]:
# Read the raw text from your local file
with open("wikitext_train.txt", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

# Clean the data: remove blanks and skip lines containing '=' (Wikipedia headers)
clean_sentences = []
for line in raw_lines:
    stripped_line = line.strip()
    
    # Keep the line only if it is not empty and not a section header
    if stripped_line and not stripped_line.startswith('='):
        clean_sentences.append(stripped_line)

# Peek at the results to verify
print(f"Total clean sentences extracted: {len(clean_sentences)}\n")
print("--- Peeking at your clean sentences ---")
for i, sentence in enumerate(clean_sentences):
    print(f"Sentence {i}: {sentence}")

Total clean sentences extracted: 9

--- Peeking at your clean sentences ---
Sentence 0: The game was released for the PlayStation Portable in Japan .
Sentence 1: It is a tactical role @-@ playing game developed by Sega .
Sentence 2: The story follows a penal military unit known as the Nameless .
Sentence 3: The original soundtrack features music composed by Hitoshi Sakimoto .
Sentence 4: It was praised for its orchestral themes and melodic depth .
Sentence 5: A total of thirty @-@ seven tracks are included in the double album .
Sentence 6: Reviewers gave the title generally favorable scores upon release .
Sentence 7: Critics highlighted the improved mission variety and maps .
Sentence 8: However some felt the graphics did not push the system to its limits .


In [17]:
# Now let's tokenize each sentences into words

In [20]:
# 1. Gather all words from our clean sentences to build a vocabulary
all_words = []
for sentence in clean_sentences:
    # Lowercase everything and split by spaces
    words = sentence.lower().split()
    all_words.extend(words)

all_words # A list of all the words

['the',
 'game',
 'was',
 'released',
 'for',
 'the',
 'playstation',
 'portable',
 'in',
 'japan',
 '.',
 'it',
 'is',
 'a',
 'tactical',
 'role',
 '@-@',
 'playing',
 'game',
 'developed',
 'by',
 'sega',
 '.',
 'the',
 'story',
 'follows',
 'a',
 'penal',
 'military',
 'unit',
 'known',
 'as',
 'the',
 'nameless',
 '.',
 'the',
 'original',
 'soundtrack',
 'features',
 'music',
 'composed',
 'by',
 'hitoshi',
 'sakimoto',
 '.',
 'it',
 'was',
 'praised',
 'for',
 'its',
 'orchestral',
 'themes',
 'and',
 'melodic',
 'depth',
 '.',
 'a',
 'total',
 'of',
 'thirty',
 '@-@',
 'seven',
 'tracks',
 'are',
 'included',
 'in',
 'the',
 'double',
 'album',
 '.',
 'reviewers',
 'gave',
 'the',
 'title',
 'generally',
 'favorable',
 'scores',
 'upon',
 'release',
 '.',
 'critics',
 'highlighted',
 'the',
 'improved',
 'mission',
 'variety',
 'and',
 'maps',
 '.',
 'however',
 'some',
 'felt',
 'the',
 'graphics',
 'did',
 'not',
 'push',
 'the',
 'system',
 'to',
 'its',
 'limits',
 '.']

In [24]:
# 2. Count frequencies of words
word_counts = Counter(all_words)
# To count the no of occurences of that word

In [27]:
# 3. Create the word-to-index mapping (and the reverse index-to-word mapping)
# We reserve 0 for padding and 1 for unknown tokens
word2idx = {"<PAD>": 0, "<UNK>": 1}
idx2word = {0: "<PAD>", 1: "<UNK>"}

# Assign an incremental index number to each unique word found
for word, count in word_counts.items():
    if word not in word2idx:
        new_index = len(word2idx)
        word2idx[word] = new_index
        idx2word[new_index] = word

vocab_size = len(word2idx)

In [30]:
# 4. Peek and verify your mapping
print(f"Total Unique Vocabulary Size: {vocab_size} words.\n")
print("--- Peeking at the first 10 entries in word2idx ---")
for word in list(word2idx.keys())[:10]:
    print(f"Word: '{word}' -> Index ID: {word2idx[word]}")

Total Unique Vocabulary Size: 77 words.

--- Peeking at the first 10 entries in word2idx ---
Word: '<PAD>' -> Index ID: 0
Word: '<UNK>' -> Index ID: 1
Word: 'the' -> Index ID: 2
Word: 'game' -> Index ID: 3
Word: 'was' -> Index ID: 4
Word: 'released' -> Index ID: 5
Word: 'for' -> Index ID: 6
Word: 'playstation' -> Index ID: 7
Word: 'portable' -> Index ID: 8
Word: 'in' -> Index ID: 9


In [ ]:
# Now that we have a hash for word2idx and idx2word we make a n-gram generator which is basically : 
"""
For example, the sentence "the game was" gets split into:
    Input: ['the'] → Target: 'game'
    Input: ['the', 'game'] → Target: 'was
    
"""

In [32]:
input_sequences = []

# Loop through each clean sentence
for sentence in clean_sentences:
    # Convert words to numerical IDs
    tokenized_sentence = [word2idx[w] for w in sentence.lower().split()] # Lower and split to avoid duplicates when searching
    
    # Create growing n-gram sequences
    for i in range(1, len(tokenized_sentence)):
        n_gram_sequence = tokenized_sentence[:i+1] # We dont add the final word becuase that will be the o/p for each n-gram
        input_sequences.append(n_gram_sequence)

# Verify the sequence generation
print(f"Total N-Gram sequences generated: {len(input_sequences)}\n")
print("--- Peeking at the first 3 generated numeric sequences ---")
for idx in range(3):
    print(f"Sequence {idx}: {input_sequences[idx]}")


Total N-Gram sequences generated: 94

--- Peeking at the first 3 generated numeric sequences ---
Sequence 0: [2, 3]
Sequence 1: [2, 3, 4]
Sequence 2: [2, 3, 4, 5]


In [35]:
# From the below word2idx you can see that  : 
# [2,3] = "the","game" and they will predict the next word "was"
# [2,3,4] = "the","game","was"  will together predict the next word "released"

# And this continues 

In [33]:
word2idx

{'<PAD>': 0,
 '<UNK>': 1,
 'the': 2,
 'game': 3,
 'was': 4,
 'released': 5,
 'for': 6,
 'playstation': 7,
 'portable': 8,
 'in': 9,
 'japan': 10,
 '.': 11,
 'it': 12,
 'is': 13,
 'a': 14,
 'tactical': 15,
 'role': 16,
 '@-@': 17,
 'playing': 18,
 'developed': 19,
 'by': 20,
 'sega': 21,
 'story': 22,
 'follows': 23,
 'penal': 24,
 'military': 25,
 'unit': 26,
 'known': 27,
 'as': 28,
 'nameless': 29,
 'original': 30,
 'soundtrack': 31,
 'features': 32,
 'music': 33,
 'composed': 34,
 'hitoshi': 35,
 'sakimoto': 36,
 'praised': 37,
 'its': 38,
 'orchestral': 39,
 'themes': 40,
 'and': 41,
 'melodic': 42,
 'depth': 43,
 'total': 44,
 'of': 45,
 'thirty': 46,
 'seven': 47,
 'tracks': 48,
 'are': 49,
 'included': 50,
 'double': 51,
 'album': 52,
 'reviewers': 53,
 'gave': 54,
 'title': 55,
 'generally': 56,
 'favorable': 57,
 'scores': 58,
 'upon': 59,
 'release': 60,
 'critics': 61,
 'highlighted': 62,
 'improved': 63,
 'mission': 64,
 'variety': 65,
 'maps': 66,
 'however': 67,
 'some': 

In [36]:
# Now let's create X and y for our model to train upon

In [37]:
# 1. Find the length of the longest sequence in our dataset ~ Used for padding 
max_len = max([len(seq) for seq in input_sequences])
print(f"Maximum sequence length found: {max_len}")


# 2. Pre-pad every sequence with zeros so they all match max_len
padded_sequences = []
for seq in input_sequences:
    padding_amount = max_len - len(seq)
    # Put zeros at the beginning, then append the original sequence
    padded_seq = [0] * padding_amount + seq
    padded_sequences.append(padded_seq)


# Convert our list of lists into a standard NumPy array for easy slicing
padded_sequences = np.array(padded_sequences)


# 3. Slice the data into Inputs (X) and Targets (y)
# X takes all columns except the very last one
X_np = padded_sequences[:, :-1]
# y takes only the very last column (the word to predict)
y_np = padded_sequences[:, -1]


# 4. Convert directly into PyTorch long tensors # IMP
X_tensor = torch.tensor(X_np, dtype=torch.long)
y_tensor = torch.tensor(y_np, dtype=torch.long)

print(f"Inputs tensor shape (X): {X_tensor.shape}")
print(f"Targets tensor shape (y): {y_tensor.shape}\n")

print("--- Peeking at padded Sequence 0 ---")
print(f"Full Padded Sequence: {padded_sequences[0]}")
print(f"Input Context (X[0]):  {X_tensor[0]}")
print(f"Target Label (y[0]):   {y_tensor[0].item()} (Which corresponds to: '{idx2word[y_tensor[0].item()]}')")


Maximum sequence length found: 14
Inputs tensor shape (X): torch.Size([94, 13])
Targets tensor shape (y): torch.Size([94])

--- Peeking at padded Sequence 0 ---
Full Padded Sequence: [0 0 0 0 0 0 0 0 0 0 0 0 2 3]
Input Context (X[0]):  tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2])
Target Label (y[0]):   3 (Which corresponds to: 'game')


In [38]:
# 1. Custom Dataset Wrapper
class NextWordDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X = X_data
        self.y = y_data
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 2. Instantiate our dataset object
text_dataset = NextWordDataset(X_tensor, y_tensor)

# 3. Create the DataLoader with a batch size of 16
# We set shuffle=True so the network doesn't accidentally memorize the sentence order
batch_size = 16
text_dataloader = DataLoader(text_dataset, batch_size=batch_size, shuffle=True)

# 4. Peek inside the first batch to verify mechanics
first_batch_X, first_batch_y = next(iter(text_dataloader))

print(f"Total dataset samples: {len(text_dataset)}")
print(f"Number of batches per epoch: {len(text_dataloader)}")
print(f"Batch X Shape: {first_batch_X.shape} (16 rows of length 13)")
print(f"Batch y Shape: {first_batch_y.shape} (16 raw class label indices)")


Total dataset samples: 94
Number of batches per epoch: 6
Batch X Shape: torch.Size([16, 13]) (16 rows of length 13)
Batch y Shape: torch.Size([16]) (16 raw class label indices)


In [39]:
# Now creating the stacked lstm model 

class StackedLSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2):
        super(StackedLSTMModel, self).__init__()
        
        # 1. Word Embedding Layer (ignoring padding calculations at Index 0)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 2. Native Multi-layer Stacked LSTM
        # batch_first=True tells PyTorch that our inputs look like (batch_size, sequence_length, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        
        # 3. Dense Fully-Connected Layer outputting scores for all vocabulary slots
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x):
        # Pass input IDs through the embedding layer
        embedded = self.embedding(x)
        
        # Pass through the stacked LSTM layers
        # lstm_out contains all hidden states across time; we only need the final step's output
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        
        # Extract the very last hidden state vector out of the time sequence sequence
        final_time_step_out = lstm_out[:, -1, :]
        
        # Output un-normalized log probability distributions (logits)
        logits = self.fc(final_time_step_out)
        return logits

# Initialize configuration parameters
EMBEDDING_DIM = 100
HIDDEN_DIM = 150
NUM_LAYERS = 2

# Instantiate the model architecture
model = StackedLSTMModel(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
print(model)


StackedLSTMModel(
  (embedding): Embedding(77, 100, padding_idx=0)
  (lstm): LSTM(100, 150, num_layers=2, batch_first=True)
  (fc): Linear(in_features=150, out_features=77, bias=True)
)


In [40]:
# Model settings and hardware setup

# 1. Select the fastest execution engine available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sending network computations to: {device}")

# Move our complete model model to the active hardware device memory space
model = model.to(device)

# 2. Setup Loss Criterion (CrossEntropy automatically applies Softmax optimization internally)
criterion = nn.CrossEntropyLoss()

# 3. Setup Gradient Descent Weight Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.005)


Sending network computations to: cpu
